# 0.36 — Ranking by density, and whether better words help

Two changes tested here:

1. **the ranking rule** — applied in `theme_basket.py`
2. **the vocabulary** — alternative word sets for each theme

Baskets are the top 10 companies. Themes are pinned in `code/themes/`.

In [1]:
import json
from pathlib import Path

import pandas as pd

CODE = Path.cwd().parent
PROC = CODE / "data" / "processed"
pd.set_option("display.width", 220)

TOP = 10
import sys; sys.path.insert(0, str(CODE))
import theme_basket as tb

load  = lambda n: pd.read_parquet(PROC / f"theme_basket_{n}.parquet")
meta  = lambda n: json.loads((PROC / f"theme_basket_{n}_manifest.json").read_text())
vocab = lambda n: meta(n)["vocab"]

def top(name, n=TOP):
    '''Basket as saved — i.e. under the new rule.'''
    return load(name).head(n)

def old_rule(name, n=TOP):
    '''What the old rule (breadth first) would have returned from the same data.'''
    return load(name).sort_values(["n_terms_hit", "per_1k_words"], ascending=False).head(n)

## 1 · The change

```python
# before                                  # after
sort_values(["n_terms_hit",               sort_values(["_floor",            # total >= 10 hits
              "per_1k_words"])                          "per_1k_words",     # density leads
                                                        "n_terms_hit"])     # breadth breaks ties
```

Density leads because a company that is *about* a theme repeats a few of its words, while one that
brushes past it touches many words once. The floor exists because density is a ratio: without it,
two mentions in a 1,000-word filing outrank 269 in 53,000.

`SCORE_RULE` is bumped to `fts-density-v2`, so old baskets no longer certify as current.

## 2 · What it changes — genAI (2023-01-17)

In [2]:
G = "genai_2023-01-17"
pd.DataFrame({"old rule (breadth)": old_rule(G).ticker.values,
              "new rule (density)": top(G).ticker.values},
             index=range(1, TOP + 1))

,old rule (breadth),new rule (density)
1,NVDA,GOOG
2,TRIP,NVDA
3,GOOG,TJX
4,MYPS,MYPS
5,APPS,BKNG
6,EXPE,OKLO
7,EBAY,AMD
8,CRNC,SIRI
9,FFAI,TRIP
10,BKNG,CSPI


## 3 · What it changes — quantum (2022-09-30)

In [3]:
Q = "quantum_2022-09-30"
pd.DataFrame({"old rule (breadth)": old_rule(Q).ticker.values,
              "new rule (density)": top(Q).ticker.values},
             index=range(1, TOP + 1))

,old rule (breadth),new rule (density)
1,IONQ,QUBT
2,QUBT,IONQ
3,RGTI,QBTS
4,QBTS,RGTI
5,LODE,QSI
6,IBM,QMCO
7,RBBN,GEOS
8,BZAI,BZAI
9,MRSH,LODE
10,AMPG,AMPG


## 4 · Alternative vocabularies

Same date, same code, different word sets.

In [4]:
VARIANTS = {
    "genai_2023-01-17": "as detected (7 of 13 words are company names)",
    "genai-names":      "company names only — a pure lookup",
    "genai-concepts":   "concepts only — no company named",
    "quantum_2022-09-30": "as detected",
    "quantum-pure":     "without the bare word 'quantum'",
    "quantum-wide":     "more quantum-technology concepts",
}
rows = []
for name, why in VARIANTS.items():
    b = load(name)
    rows.append({"variant": name, "words": len(vocab(name)), "companies": len(b),
                 "top 10": ", ".join(b.ticker.head(TOP)), "note": why})
pd.DataFrame(rows).set_index("variant")

,words,companies,top 10,note
variant,,,,
genai_2023-01-17,13,98,"GOOG, NVDA, TJX, MYPS, BKNG, OKLO, AMD, SIRI, ...",as detected (7 of 13 words are company names)
genai-names,7,74,"GOOG, NVDA, MYPS, BKNG, AMD, TRIP, CSPI, OOMA,...",company names only — a pure lookup
genai-concepts,13,69,"LPSN, TJX, OKLO, BC, CAPR, AMBA, RARE, JAZZ, S...",concepts only — no company named
quantum_2022-09-30,12,153,"QUBT, IONQ, QBTS, RGTI, QSI, QMCO, GEOS, BZAI,...",as detected
quantum-pure,11,35,"QUBT, IONQ, RGTI, QBTS, AMPG, LODE, HON, IBM, ...",without the bare word 'quantum'
quantum-wide,20,157,"QUBT, IONQ, QBTS, RGTI, QSI, QMCO, BZAI, GEOS,...",more quantum-technology concepts


## 5 · The variants side by side

In [5]:
for name, why in VARIANTS.items():
    b = top(name)
    print(f"=== {name}  —  {why}")
    if b.empty:
        print("    EMPTY — no company in the universe mentions these words at this date\n")
        continue
    print(b[["ticker", "company", "total", "n_terms_hit", "per_1k_words"]].to_string(index=False), "\n")

=== genai_2023-01-17  —  as detected (7 of 13 words are company names)
ticker                    company  total  n_terms_hit  per_1k_words
  GOOG              Alphabet Inc.    269            4         5.088
  NVDA                NVIDIA CORP    613            5         4.213
   TJX     TJX COMPANIES INC /DE/     39            2         0.738
  MYPS          PLAYSTUDIOS, Inc.    129            4         0.602
  BKNG      Booking Holdings Inc.     49            3         0.599
  OKLO                  Oklo Inc.     61            2         0.557
   AMD ADVANCED MICRO DEVICES INC     65            3         0.391
  SIRI    SIRIUS XM HOLDINGS INC.     22            3         0.353
  TRIP          TripAdvisor, Inc.     24            5         0.319
  CSPI               CSP INC /MA/     13            3         0.306 

=== genai-names  —  company names only — a pure lookup
ticker                    company  total  n_terms_hit  per_1k_words
  GOOG              Alphabet Inc.    269            4   

## 6 · What the variants show

**quantum — a better vocabulary exists.** Drop the bare word `quantum` and the basket goes from
153 companies to 35, keeps the same top four, and replaces three false positives with two real ones:

In [6]:
b, v = load("quantum_2022-09-30"), vocab("quantum_2022-09-30")
b.loc[b.ticker.isin(["QSI", "QMCO", "GEOS"]), ["ticker", "company"] + v].set_index("ticker").T.pipe(
    lambda d: d[d.sum(axis=1, numeric_only=True).ne(0) | (d.index == "company")])

ticker,QSI,QMCO,GEOS
company,Quantum-Si Inc,QUANTUM CORP /DE/,GEOSPACE TECHNOLOGIES CORP


Quantum-Si, Quantum Corp and Geospace score **entirely** on the word `quantum` — it is in their
company or product name, and none of them does quantum computing. Removing that one word drops them
and lets Honeywell and IBM — both with real quantum programmes — into the top ten.

**Recommendation: use `quantum-pure` as the quantum theme.**

**genAI — a better vocabulary does not exist at this date.**

- `genai-names` (company names only) returns almost the same basket as the full vocabulary.
  The genAI basket *is* a name lookup.
- `genai-concepts` finds almost nothing: `chatgpt` appears in 0 filings, `openai` in 2,
  `generative ai` in 6. Its one genuine find is LivePerson (conversational AI).

No word choice repairs this, because the words are not in the filings yet. That is not a vocabulary
problem — it is the wrong source of text, and it is the next step.

---
## Summary

| | density ranking | better words |
|---|---|---|
| quantum | top 4 unchanged — control passes | **yes** — drop the bare word `quantum` |
| genAI | GOOG and NVDA rise, TripAdvisor falls 2 → 9 | **no** — the concepts are not in filings yet |

Still open: whether the new ordering earns more than the old one. That is the backtest.

---
# Method 3 — weight words by how rare they are

Counting treats every theme word as equal evidence. It is not:

- `jpmorgan` is in 4,385 filings and `google` in 1,699 — background noise in any US filing.
  `openai` is in 2. **Weight each hit by log(corpus / how many filings contain the word).**
- A company scoring entirely on *one* word is almost always a false positive.
  **Require hits on at least two words specific enough to shortlist on.**

Both numbers are already saved by every build, so this is a re-ranking — no EDGAR, no rebuild.

In [7]:
third = lambda n: tb.rank_specificity(load(n), meta(n))

t = third(G)
t[["ticker", "company", "total", "n_specific", "per_1k_words", "weighted"]].head(TOP)

,ticker,company,total,n_specific,per_1k_words,weighted
0,NVDA,NVIDIA CORP,613,3,4.213,23.095601
1,GOOG,Alphabet Inc.,269,2,5.088,14.763504
2,OKLO,Oklo Inc.,61,2,0.557,3.430292
3,TRIP,"TripAdvisor, Inc.",24,2,0.319,1.039700
4,EXPE,"Expedia Group, Inc.",18,2,0.244,0.785050
5,ABNB,"Airbnb, Inc.",19,2,0.185,0.655645
6,EBAY,EBAY INC,11,2,0.172,0.607183
7,CRNC,Cerence Inc.,4,2,0.058,0.262852
8,FFAI,FARADAY FUTURE INTELLIGENT ELECTRIC INC.,7,2,0.055,0.258584


## The three methods side by side

In [8]:
def compare(name):
    a = old_rule(name).ticker.values
    b = top(name).ticker.values
    c = third(name).ticker.head(TOP).values
    c = list(c) + [""] * (TOP - len(c))
    return pd.DataFrame({"1 · count (breadth)": a, "2 · density": b, "3 · rarity-weighted": c},
                        index=range(1, TOP + 1))

print("genAI 2023-01-17"); display(compare(G))
print("quantum 2022-09-30"); display(compare(Q))

genAI 2023-01-17


,1 · count (breadth),2 · density,3 · rarity-weighted
1,NVDA,GOOG,NVDA
2,TRIP,NVDA,GOOG
3,GOOG,TJX,OKLO
4,MYPS,MYPS,TRIP
5,APPS,BKNG,EXPE
6,EXPE,OKLO,ABNB
7,EBAY,AMD,EBAY
8,CRNC,SIRI,CRNC
9,FFAI,TRIP,FFAI
10,BKNG,CSPI,


quantum 2022-09-30


,1 · count (breadth),2 · density,3 · rarity-weighted
1,IONQ,QUBT,QUBT
2,QUBT,IONQ,IONQ
3,RGTI,QBTS,QBTS
4,QBTS,RGTI,RGTI
5,LODE,QSI,BZAI
6,IBM,QMCO,LODE
7,RBBN,GEOS,MRSH
8,BZAI,BZAI,AMPG
9,MRSH,LODE,HON
10,AMPG,AMPG,IBM


## What method 3 changes

**quantum — it finds the right basket without editing the vocabulary.**
Quantum-Si, Quantum Corp and Geospace disappear: they scored only on the word `quantum`, which is
in their company names. Honeywell and IBM take their place. This is the same basket the hand-written
`quantum-pure` word list produced — reached automatically, from the full vocabulary.

**genAI — it removes the noise but cannot invent a theme.**
TJX (`ernie` = its CEO), Ultragenyx and Jazz are gone, and NVIDIA and Alphabet lead. But only 9 of 98
companies clear the two-word gate, and below rank three the survivors hold 2 mentions of `alibaba`
or `baidu`. Nothing is there to find: the theme's own words are not in the filings yet.

The one free parameter is the corpus size in the IDF. The top ten is stable from 5,000 to 200,000,
so it is not doing the work — the ranking is.